# Football Scouting Tool — Percentile Radars & Player Similarity

**Goal**: turn raw per-90 stats into two things scouts actually use —
**percentile radars** ("how does this player rank against their position
peers, on the metrics that matter for that position?") and a
**similarity search** ("who plays like this player?") — plus a look at
whether statistically distinct *playing-style archetypes* exist within a
position at all.

**Dataset**: FBref "Big 5 European Leagues" advanced season stats
(standard, shooting, passing, possession, defense, goal/shot-creating
actions, miscellaneous), via the [`worldfootballR_data`](https://github.com/JaseZiv/worldfootballR_data)
project. Full feature-engineering pipeline in `src/build_features.py` —
this notebook loads its output.


In [1]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
import plotly.express as px

from similarity import PlayerSimilarity, FEATURE_COLS
from radar import compute_percentiles, player_radar, RADAR_TEMPLATES
from style_clusters import sweep_k, cluster_position_group
from sklearn.preprocessing import StandardScaler

df = pd.read_parquet("../data/processed/players_2022.parquet")
print(f"{len(df):,} players — {df['position'].value_counts().to_dict()}")


1,499 players — {'DF': 637, 'MF': 597, 'FW': 265}


## 1. A data-quality check worth being upfront about

This dataset's most recent season (2022-23) is actually an **incomplete
scrape** — it tops out at 1,170 minutes played per player, about a third
of a full season, instead of the ~3,400 a ever-present starter reaches.
Using it as-is would have quietly made every player in the "current"
season look like a squad-rotation player. The **2021-22 season is the
most recent one with a complete scrape** (max minutes ≈3,420, matching a
real full season), so that's what this project uses throughout.


In [2]:
from load_data import load_table
standard = load_table("big5_player_standard")
print(standard.groupby("Season_End_Year")["Min_Playing"].max())


Season_End_Year
2010    3420.0
2011    3420.0
2012    3420.0
2013    3420.0
2014    3420.0
2015    3420.0
2016    3420.0
2017    3420.0
2018    3420.0
2019    3420.0
2020    3420.0
2021    3420.0
2022    3420.0
2023    1170.0
Name: Min_Playing, dtype: float64


## 2. Feature engineering: raw counts to comparable per-90 rates

Most FBref columns are season totals, which aren't comparable across
players with different playing time. Every rate feature here (`*_p90`) is
computed by hand as `raw_count / (minutes_played / 90)`, restricted to
players with **≥900 minutes** in the season (~10 full matches — the usual
floor for including a player in per-90 comparisons at all, otherwise a
player's one hot game skews their rate wildly). Goalkeepers are excluded
entirely — their metrics live in a different world.

Full curated feature list: `src/similarity.py::FEATURE_COLS` (30 per-90
rates and percentages across attacking, passing, carrying, and defending).


In [3]:
df[["player","club","league","position","age","minutes"] + FEATURE_COLS[:6]].sample(5, random_state=3)


,player,club,league,position,age,minutes,goals_p90,assists_p90,npxg_p90,xag_p90,shots_p90,sot_pct
851,Genki Haraguchi,Union Berlin,Bundesliga,MF,30,1755.0,0.102564,0.307692,0.153846,0.102564,1.487179,20.7
816,Roberto Pereyra,Udinese,Serie A,MF,30,1794.0,0.150502,0.250836,0.190635,0.215719,1.404682,25.0
919,Matteo Politano,Napoli,Serie A,MF,27,1650.0,0.163636,0.218182,0.141818,0.207273,2.290909,26.2
1042,Wylan Cyprien,Nantes,Ligue 1,MF,26,1490.0,0.120805,0.181208,0.072483,0.205369,1.268456,38.1
581,Yves Bissouma,Brighton,Premier League,MF,24,2111.0,0.042634,0.085268,0.055424,0.034107,0.895310,23.8
